In [1]:
from google.colab import drive
drive.mount("/content/drive")


Mounted at /content/drive


In [2]:
%cd "/content/drive/MyDrive/Colab Notebooks/LLM Project"

/content/drive/MyDrive/Colab Notebooks/LLM Project


In [3]:
!pip install zarr numcodecs


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 255.4/255.4 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 27.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.5/53.5 kB 5.9 MB/s eta 0:00:00


In [4]:
import zarr
import numpy as np
from numpy.linalg import inv
import gc


# -----------------------
# 1. Load MRIO data
# -----------------------
store = zarr.open("2022.zarr", mode="r")

T = store["T"][:]   # inter-industry transactions
Y = store["Y"][:]   # final demand
Q = store["Q"][:]   # environmental extensions

regions_in  = store["input_region"][:]
regions_out = store["output_region"][:]
sectors_in  = store["input_sector"][:]
sectors_out = store["output_sector"][:]
indicators  = store["environmental_indicator"][:]

# -----------------------
# 2. Get dimensions
# -----------------------
n_reg = len(regions_in)    # number of regions
n_sec = len(sectors_in)    # number of sectors per region
n_tot = n_reg * n_sec      # total (region × sector)
n_env = len(indicators)    # number of environmental indicators

print(f"Regions: {n_reg}, Sectors: {n_sec}, Total: {n_tot}, Indicators: {n_env}")

# -----------------------
# 3. Reshape into 2D form
# -----------------------

# reshape T into Z, then free T
Z = T.reshape(n_tot, n_tot)
del T
gc.collect()

# reshape Y into 2D, aggregate, then free Y
y = Y.reshape(n_tot, n_reg).sum(1)
del Y
gc.collect()

# reshape Q into F, then free Q
F = Q.reshape(n_env, n_tot)
del Q
gc.collect()

print("Z, y, F ready for MRIO calculations")
# -----------------------
# 4. Load D matrix result
# -----------------------

z = zarr.open("Environmental_Impact_2022.zarr", mode="r")
print("Keys at root:", list(z.keys()))

# Open the Environmental_Impact_3D array
D = z["Environmental_Impact_3D"]
print("Original D shape:", D.shape)



Regions: 189, Sectors: 163, Total: 30807, Indicators: 19
Z, y, F ready for MRIO calculations
Keys at root: ['Environmental_Impact_3D']
Original D shape: (19, 30807, 30807)


In [5]:
import pandas as pd

# Load the sector file
sectors_df = pd.read_excel("REX3_grouping.xlsx", sheet_name="Sectors")

# Define the food category IDs
food_sector_ids = {
    "Crops": [1, 2, 3, 4, 5, 6, 7, 8],
    "Livestock": [9, 10, 11, 12, 13, 14],
    "Fishing": [19, 45],
    "FoodProcessing": list(range(35, 44)),
    "Beverages": [44],
}

# Create a list to hold the formatted data
formatted_data = []

# Iterate through each food category and its IDs
for category, ids in food_sector_ids.items():
    # Filter the DataFrame for sectors in the current category
    category_sectors = sectors_df[sectors_df['No Sector REX3'].isin(ids)]

    # Iterate through the filtered rows and create the dictionary structure
    for index, row in category_sectors.iterrows():
        sector_dict = {
            "Category": category,
            "Sector_ID": row['No Sector REX3'],
            "Sector_Name": row['Sector name']
        }
        formatted_data.append(sector_dict)

# Convert the list of dictionaries into a DataFrame
sector_target = pd.DataFrame(formatted_data)

# Print the final DataFrame
display(sector_target)

,Category,Sector_ID,Sector_Name
0,Crops,1,Cultivation of paddy rice
1,Crops,2,Cultivation of wheat
2,Crops,3,Cultivation of cereal grains nec
3,Crops,4,"Cultivation of vegetables, fruit, nuts"
4,Crops,5,Cultivation of oil seeds
5,Crops,6,"Cultivation of sugar cane, sugar beet"
6,Crops,7,Cultivation of plant-based fibers
7,Crops,8,Cultivation of crops nec
8,Livestock,9,Cattle farming
9,Livestock,10,Pigs farming


In [19]:
indicator_map = {
    0: "CO2",
    1: "CH4",
    2: "N2O",
    3: "Other GHG",
    4: "GHG total",
    5: "Land use",
    6: "Water use",
    # ...
}

target_indicators = ["GHG total", "Land use", "Water use"]
target_indices = [i for i, name in indicator_map.items() if name in target_indicators]
print("Target indicator indices:", target_indices)


Target indicator indices: [4, 5, 6]


In [25]:
#FOR GHG total
import numpy as np
import pandas as pd
import gc

# Inputs you already have:
# D: (n_env, n_tot, n_tot)  -> Environmental_Impact_3D
# indicators: np.ndarray of indicator names (len = n_env), e.g. ["CO2","GHG",...]
# regions: list of region codes (len = n_reg)
# n_reg: number of regions
# n_sec: number of sectors per region (so n_tot = n_reg * n_sec)

# food categories (per-sector IDs, repeated in every region)
# Define the food category IDs
food_sector_ids = {
    "Crops": [1, 2, 3, 4, 5, 6, 7, 8],
    "Livestock": [9, 10, 11, 12, 13, 14],
    "Fishing": [19, 45],
    "FoodProcessing": list(range(35, 44)),
    "Beverages": [44],
}

# Flat list of food sector IDs
food_ids = sorted(set([s for ids in food_sector_ids.values() for s in ids]))

# Load region mapping from Excel
region_map = pd.read_excel("REX3_grouping.xlsx", sheet_name="Regions")

# Assume first column = ID, second column = Name
region_id_to_name = dict(zip(region_map.iloc[:,0], region_map.iloc[:,1]))


# 1) Pick GHG total indicator from D matrix

D_co2 = D[4]  # shape (n_tot contries, n_tot sectors)

# 2) Build global indices for food sectors across all regions ()
# Define regions list (replace with actual codes from your MRIO metadata)
regions = list(range(1, 189))  # example only

food_positions = np.fromiter(
    (r * n_sec + s for r in range(n_reg) for s in food_ids),
    dtype=np.int64,
    count=n_reg * len(food_ids)
)

# 3) Restrict to food-only block (producer×consumer among food sectors)
E_food = D_co2[np.ix_(food_positions, food_positions)]
del D_co2
gc.collect()

# Optional: if you want the 4D tensor (pr, ps, cr, cs)
E_4d = E_food.reshape(n_reg, len(food_ids), n_reg, len(food_ids))

# 4) Top-50 flows by value (no Python loops; stays memory-light)
flat = E_food.ravel()
k = 50
idx = np.argpartition(flat, -k)[-k:]       # get indices of top-k
top_vals = flat[idx]
pi, pj = np.unravel_index(idx, E_food.shape)

# Map back to original global producer/consumer indices
prod_global = food_positions[pi]
cons_global = food_positions[pj]

# Derive region & sector indices
prod_reg = (prod_global // n_sec).astype(int)
prod_sec = (prod_global %  n_sec).astype(int)
cons_reg = (cons_global // n_sec).astype(int)
cons_sec = (cons_global %  n_sec).astype(int)

# Map sector -> category (for nicer labels; falls back to the sector ID if not in a category)
sec_to_cat = np.array(["Other"] * n_sec, dtype=object)
for cat, ids in food_sector_ids.items():
    for s in ids:
        if s < n_sec:  # guard
            sec_to_cat[s] = cat

df_top = pd.DataFrame({
    "Producer_region_id": np.array(regions)[prod_reg],
    "Producer_sector_id": prod_sec,
    "Producer_category": sec_to_cat[prod_sec],
    "Consumer_region_id": np.array(regions)[cons_reg],
    "Consumer_sector_id": cons_sec,
    "Consumer_category": sec_to_cat[cons_sec],
    "Indicator": "GHG total",
    "Value": top_vals
})

# Map region IDs to names using region_id_to_name
df_top["Producer_region"] = df_top["Producer_region_id"].map(region_id_to_name)
df_top["Consumer_region"] = df_top["Consumer_region_id"].map(region_id_to_name)

# Sort by value
df_top = df_top.sort_values("Value", ascending=False).reset_index(drop=True)

display(df_top.head(50))



,Producer_region_id,Producer_sector_id,Producer_category,Consumer_region_id,Consumer_sector_id,Consumer_category,Indicator,Value,Producer_region,Consumer_region
0,100,3,Crops,31,44,Beverages,GHG total,4.703245e+12,Haiti,China
1,45,44,Beverages,31,44,Beverages,GHG total,3.630950e+12,Afghanistan,China
2,100,3,Crops,29,42,FoodProcessing,GHG total,3.053634e+12,Haiti,USA
3,100,2,Crops,31,44,Beverages,GHG total,2.463607e+12,Haiti,China
4,45,44,Beverages,29,42,FoodProcessing,GHG total,2.357435e+12,Afghanistan,USA
5,100,2,Crops,29,42,FoodProcessing,GHG total,1.599525e+12,Haiti,USA
6,100,3,Crops,30,42,FoodProcessing,GHG total,1.299916e+12,Haiti,Japan
7,100,3,Crops,31,3,Crops,GHG total,1.274938e+12,Haiti,China
8,100,3,Crops,29,43,FoodProcessing,GHG total,1.105035e+12,Haiti,USA
9,45,44,Beverages,30,42,FoodProcessing,GHG total,1.003547e+12,Afghanistan,Japan


In [28]:
#FOR Land use
import numpy as np
import pandas as pd
import gc

# Inputs you already have:
# D: (n_env, n_tot, n_tot)  -> Environmental_Impact_3D
# indicators: np.ndarray of indicator names (len = n_env), e.g. ["CO2","GHG",...]
# regions: list of region codes (len = n_reg)
# n_reg: number of regions
# n_sec: number of sectors per region (so n_tot = n_reg * n_sec)

# food categories (per-sector IDs, repeated in every region)
# Define the food category IDs
food_sector_ids = {
    "Crops": [1, 2, 3, 4, 5, 6, 7, 8],
    "Livestock": [9, 10, 11, 12, 13, 14],
    "Fishing": [19, 45],
    "FoodProcessing": list(range(35, 44)),
    "Beverages": [44],
}

# Flat list of food sector IDs
food_ids = sorted(set([s for ids in food_sector_ids.values() for s in ids]))

# Load region mapping from Excel
region_map = pd.read_excel("REX3_grouping.xlsx", sheet_name="Regions")

# Assume first column = ID, second column = Name
region_id_to_name = dict(zip(region_map.iloc[:,0], region_map.iloc[:,1]))


# 1) Pick land use indicator from D matrix

D_co2 = D[5]  # shape (n_tot contries, n_tot sectors)

# 2) Build global indices for food sectors across all regions ()
# Define regions list (replace with actual codes from your MRIO metadata)
regions = list(range(1, 189))  # example only

food_positions = np.fromiter(
    (r * n_sec + s for r in range(n_reg) for s in food_ids),
    dtype=np.int64,
    count=n_reg * len(food_ids)
)

# 3) Restrict to food-only block (producer×consumer among food sectors)
E_food = D_co2[np.ix_(food_positions, food_positions)]
del D_co2
gc.collect()

# Optional: if you want the 4D tensor (pr, ps, cr, cs)
E_4d = E_food.reshape(n_reg, len(food_ids), n_reg, len(food_ids))

# 4) Top-50 flows by value (no Python loops; stays memory-light)
flat = E_food.ravel()
k = 50
idx = np.argpartition(flat, -k)[-k:]       # get indices of top-k
top_vals = flat[idx]
pi, pj = np.unravel_index(idx, E_food.shape)

# Map back to original global producer/consumer indices
prod_global = food_positions[pi]
cons_global = food_positions[pj]

# Derive region & sector indices
prod_reg = (prod_global // n_sec).astype(int)
prod_sec = (prod_global %  n_sec).astype(int)
cons_reg = (cons_global // n_sec).astype(int)
cons_sec = (cons_global %  n_sec).astype(int)

# Map sector -> category (for nicer labels; falls back to the sector ID if not in a category)
sec_to_cat = np.array(["Other"] * n_sec, dtype=object)
for cat, ids in food_sector_ids.items():
    for s in ids:
        if s < n_sec:  # guard
            sec_to_cat[s] = cat

df_top = pd.DataFrame({
    "Producer_region_id": np.array(regions)[prod_reg],
    "Producer_sector_id": prod_sec,
    "Producer_category": sec_to_cat[prod_sec],
    "Consumer_region_id": np.array(regions)[cons_reg],
    "Consumer_sector_id": cons_sec,
    "Consumer_category": sec_to_cat[cons_sec],
    "Indicator": "Land use",
    "Value": top_vals
})

# Map region IDs to names using region_id_to_name
df_top["Producer_region"] = df_top["Producer_region_id"].map(region_id_to_name)
df_top["Consumer_region"] = df_top["Consumer_region_id"].map(region_id_to_name)

# Sort by value
df_top = df_top.sort_values("Value", ascending=False).reset_index(drop=True)

display(df_top.head(50))


,Producer_region_id,Producer_sector_id,Producer_category,Consumer_region_id,Consumer_sector_id,Consumer_category,Indicator,Value,Producer_region,Consumer_region
0,45,44,Beverages,31,44,Beverages,Land use,9.424989e+13,Afghanistan,China
1,45,44,Beverages,29,42,FoodProcessing,Land use,6.119279e+13,Afghanistan,USA
2,100,3,Crops,31,44,Beverages,Land use,2.652408e+13,Haiti,China
3,45,44,Beverages,30,42,FoodProcessing,Land use,2.604944e+13,Afghanistan,Japan
4,45,44,Beverages,31,3,Crops,Land use,2.554891e+13,Afghanistan,China
5,45,44,Beverages,29,43,FoodProcessing,Land use,2.214417e+13,Afghanistan,USA
6,45,44,Beverages,35,45,Fishing,Land use,1.906322e+13,Afghanistan,India
7,100,3,Crops,29,42,FoodProcessing,Land use,1.722106e+13,Haiti,USA
8,100,2,Crops,31,44,Beverages,Land use,1.588717e+13,Haiti,China
9,45,44,Beverages,35,13,Livestock,Land use,1.561623e+13,Afghanistan,India


In [29]:
#FOR Water use
import numpy as np
import pandas as pd
import gc

# Inputs you already have:
# D: (n_env, n_tot, n_tot)  -> Environmental_Impact_3D
# indicators: np.ndarray of indicator names (len = n_env), e.g. ["CO2","GHG",...]
# regions: list of region codes (len = n_reg)
# n_reg: number of regions
# n_sec: number of sectors per region (so n_tot = n_reg * n_sec)

# food categories (per-sector IDs, repeated in every region)
# Define the food category IDs
food_sector_ids = {
    "Crops": [1, 2, 3, 4, 5, 6, 7, 8],
    "Livestock": [9, 10, 11, 12, 13, 14],
    "Fishing": [19, 45],
    "FoodProcessing": list(range(35, 44)),
    "Beverages": [44],
}

# Flat list of food sector IDs
food_ids = sorted(set([s for ids in food_sector_ids.values() for s in ids]))

# Load region mapping from Excel
region_map = pd.read_excel("REX3_grouping.xlsx", sheet_name="Regions")

# Assume first column = ID, second column = Name
region_id_to_name = dict(zip(region_map.iloc[:,0], region_map.iloc[:,1]))


# 1) Pick land use indicator from D matrix

D_co2 = D[6]  # shape (n_tot contries, n_tot sectors)

# 2) Build global indices for food sectors across all regions ()
# Define regions list (replace with actual codes from your MRIO metadata)
regions = list(range(1, 189))  # example only

food_positions = np.fromiter(
    (r * n_sec + s for r in range(n_reg) for s in food_ids),
    dtype=np.int64,
    count=n_reg * len(food_ids)
)

# 3) Restrict to food-only block (producer×consumer among food sectors)
E_food = D_co2[np.ix_(food_positions, food_positions)]
del D_co2
gc.collect()

# Optional: if you want the 4D tensor (pr, ps, cr, cs)
E_4d = E_food.reshape(n_reg, len(food_ids), n_reg, len(food_ids))

# 4) Top-50 flows by value (no Python loops; stays memory-light)
flat = E_food.ravel()
k = 50
idx = np.argpartition(flat, -k)[-k:]       # get indices of top-k
top_vals = flat[idx]
pi, pj = np.unravel_index(idx, E_food.shape)

# Map back to original global producer/consumer indices
prod_global = food_positions[pi]
cons_global = food_positions[pj]

# Derive region & sector indices
prod_reg = (prod_global // n_sec).astype(int)
prod_sec = (prod_global %  n_sec).astype(int)
cons_reg = (cons_global // n_sec).astype(int)
cons_sec = (cons_global %  n_sec).astype(int)

# Map sector -> category (for nicer labels; falls back to the sector ID if not in a category)
sec_to_cat = np.array(["Other"] * n_sec, dtype=object)
for cat, ids in food_sector_ids.items():
    for s in ids:
        if s < n_sec:  # guard
            sec_to_cat[s] = cat

df_top = pd.DataFrame({
    "Producer_region_id": np.array(regions)[prod_reg],
    "Producer_sector_id": prod_sec,
    "Producer_category": sec_to_cat[prod_sec],
    "Consumer_region_id": np.array(regions)[cons_reg],
    "Consumer_sector_id": cons_sec,
    "Consumer_category": sec_to_cat[cons_sec],
    "Indicator": "Water use",
    "Value": top_vals
})

# Map region IDs to names using region_id_to_name
df_top["Producer_region"] = df_top["Producer_region_id"].map(region_id_to_name)
df_top["Consumer_region"] = df_top["Consumer_region_id"].map(region_id_to_name)

# Sort by value
df_top = df_top.sort_values("Value", ascending=False).reset_index(drop=True)

display(df_top.head(50))


,Producer_region_id,Producer_sector_id,Producer_category,Consumer_region_id,Consumer_sector_id,Consumer_category,Indicator,Value,Producer_region,Consumer_region
0,127,6,Crops,31,44,Beverages,Water use,1.656242e+15,Benin,China
1,127,6,Crops,29,42,FoodProcessing,Water use,1.075334e+15,Benin,USA
2,127,6,Crops,30,42,FoodProcessing,Water use,4.577639e+14,Benin,Japan
3,127,6,Crops,31,3,Crops,Water use,4.489682e+14,Benin,China
4,127,6,Crops,29,43,FoodProcessing,Water use,3.891369e+14,Benin,USA
5,127,6,Crops,35,45,Fishing,Water use,3.349958e+14,Benin,India
6,127,6,Crops,35,13,Livestock,Water use,2.744223e+14,Benin,India
7,126,2,Crops,31,44,Beverages,Water use,2.711642e+14,Angola,China
8,127,6,Crops,35,3,Crops,Water use,2.615821e+14,Benin,India
9,127,6,Crops,29,3,Crops,Water use,2.516247e+14,Benin,USA
